# Particle Filter for Wellbore TVT Prediction — Plateau Scale Average

**Approach.** A Bayesian particle filter that tracks True Vertical Thickness (TVT) along a
horizontal well by matching its gamma-ray (GR) log to the typewell GR(TVT) profile, with a
motion model whose drift rate is estimated from the known lead-in rows. Pure GR-tracking — it
uses only MD/X/Y/Z/GR/TVT_input (no formation-contact / target-derived columns).

**Attribution.** The particle-filter implementation is adapted from the public kernel
`yaroslavkholmirzayev/reproduce-strongest-reference-aeroridge-v34`. This notebook isolates its
GR-tracking PF path and contributes the plateau-scale average around the current public-best PF region.

## Parameter Study (this notebook's contribution)

Swept on held-out wells (RMSE, ft):

| Parameter | Tried | Best |
|---|---|---|
| Likelihood scale | 2,3,5,7,10,20 | **5** |
| n_particles | 500,1000,2000 | **500** (1000 worse — more particles hurt) |
| n_seeds (lik-ensemble) | 128,256 | **128** (256 worse) |
| Beam ensemble | — | much worse; every PF+beam blend worse than pure PF |
| Anchor-regularization blend | 0.8/0.7/0.5 | worse |

**Takeaway:** the *pure* PF at scale=5 / 500 particles / 128 seeds is optimal across the whole
space — more particles/seeds and any ensemble or blend strictly hurt.

In [ ]:
"""
ROGII Wellbore Geology Prediction — v12
DSC 204A Final Project

Strategy:
  - Visible training wells: physical model (RMSE ~0.007 ft)
  - Hidden test wells: PF ensemble ONLY — 128 seeds, lik-weighted (scale=5)
    init_spread=2.0 ft (wider initial particle spread)
    GR interpolated before PF (fills NaN gaps so PF always has observations)
    Local avg: 4.71 ft (vs 5.95 ft without GR interpolation)
    Key insight: 000d7d20 has 47% NaN GR in prediction — interpolation critical
"""

import os, glob, warnings, json, time
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings('ignore')

NOTEBOOK_RUN_VERSION = 'ROGII_physical_pf_plateauavg_2026_06_05'
_t0 = time.time()
def _dbg(msg):
    print(f'[{(time.time() - _t0) / 60:.1f}m] {msg}', flush=True)

PF_N_PARTICLES = int(os.environ.get('ROGII_PF_N_PARTICLES', '500'))
PF_N_SEEDS = int(os.environ.get('ROGII_PF_N_SEEDS', '128'))
PF_LIK_SCALES = [float(x) for x in os.environ.get('ROGII_PF_LIK_SCALES', '2.6875,2.71875,2.75').split(',')]
PF_LIK_SCALE = float(np.mean(PF_LIK_SCALES))
_dbg(f'=== {NOTEBOOK_RUN_VERSION} START ===')
_dbg(f'PF config: particles={PF_N_PARTICLES}, seeds={PF_N_SEEDS}, scales={PF_LIK_SCALES}')


def find_input_dir():
    override = os.environ.get('ROGII_INPUT_DIR')
    candidates = []
    if override:
        candidates.append(override)
    candidates.extend([
        '/kaggle/input/rogii-wellbore-geology-prediction',
        '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
        os.path.join(os.getcwd(), 'data'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
    ])
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, 'train')) and os.path.isdir(os.path.join(c, 'test')):
            print(f'INPUT_DIR={c}')
            return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f'Discovered INPUT_DIR={d}')
        return d
    raise FileNotFoundError('Cannot locate competition data')

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
print(f'Test wells: {TEST_WELLS}')


def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    """Conservative PF. Returns (predictions_array, total_log_likelihood)."""
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)  # wider init spread helps wells with abrupt TVT shift at PS
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking — critical for wells with high NaN fraction
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    """Run seeds once, then average likelihood-weighted ensembles across one or more scales."""
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    pred_stack = np.stack(preds, 0)
    liks   = np.array(liks)
    liks_n = liks - liks.max()
    scales = scale if isinstance(scale, (list, tuple, np.ndarray)) else [scale]
    outs = []
    for sc in scales:
        weights = np.exp(liks_n / float(sc))
        weights /= weights.sum()
        outs.append((weights[:, None] * pred_stack).sum(0))
    return np.mean(np.stack(outs, 0), axis=0)


# 14 beam configs: original 7 + 7 new ones exploring broader parameter space
BEAM_CONFIGS = [
    # Original 7 configs (from ajayrao43)
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    # 7 new configs: wider beam, different motion/error scales
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    """Vectorized beam search for TVT tracking via GR matching."""
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    """Average 14 beam-search configs."""
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
print(f'Training wells available: {len(train_wids)}')

rows = []
well_diagnostics = []
for wid in TEST_WELLS:
    print(f'\nProcessing {wid}...')
    hw_te, tw_te = load_well(wid, 'test')

    tvt_phys = None
    hw_tr    = None
    tw_tr    = None

    # Physical model for visible wells
    if False:  # FORCE PF — legit GR-tracking, NO leak (physical model uses train TVT)
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print(f'  Physical model OK')
        except Exception as e:
            print(f'  Physical model failed: {e}')
            tvt_phys = None

    # 64-seed likelihood-weighted PF ensemble
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        tvt_pf = run_pf_lik_ensemble(hw_te, tw_ref, n_particles=PF_N_PARTICLES, n_seeds=PF_N_SEEDS, scale=PF_LIK_SCALES)
        print(f'  PF {PF_N_SEEDS}-seed lik-ensemble OK')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)

    # 14-config beam ensemble
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        if tvt_phys is not None:
            # Visible well: physical model is primary (RMSE ~0.007 ft)
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            # Hidden well: PF-only (beam hurts bimodal-drift wells)
            tvt_val = float(tvt_pf[ridx])
        rows.append({'id': row['id'], 'tvt': tvt_val})
    well_diagnostics.append({
        'well': wid,
        'rows': int(len(ws)),
        'used_physical_model': bool(tvt_phys is not None),
        'pf_min': float(np.nanmin(tvt_pf)),
        'pf_max': float(np.nanmax(tvt_pf)),
        'beam_min': float(np.nanmin(tvt_beam)),
        'beam_max': float(np.nanmax(tvt_beam)),
    })
    print(f'  Added {len(ws)} rows')

submission = sample[['id']].merge(pd.DataFrame(rows), on='id', how='left')
if submission['tvt'].isna().any():
    fallback = float(np.nanmean(submission['tvt']))
    print(f"WARN: filling {int(submission['tvt'].isna().sum())} missing tvt values with {fallback:.3f}")
    submission['tvt'] = submission['tvt'].fillna(fallback)
submission.to_csv('submission.csv', index=False)

diag = {
    'run_version': NOTEBOOK_RUN_VERSION,
    'input_dir': INPUT_DIR,
    'test_wells': TEST_WELLS,
    'pf_n_particles': int(PF_N_PARTICLES),
    'pf_n_seeds': int(PF_N_SEEDS),
    'pf_lik_scales': [float(x) for x in PF_LIK_SCALES],
    'submission_rows': int(len(submission)),
    'submission_tvt_min': float(submission['tvt'].min()),
    'submission_tvt_max': float(submission['tvt'].max()),
    'submission_tvt_mean': float(submission['tvt'].mean()),
    'well_diagnostics': well_diagnostics,
}
with open('run_diagnostics_physical_pf_9150_repeat.json', 'w') as f:
    json.dump(diag, f, indent=2, sort_keys=True)

print(f'\nDone: {len(submission)} rows')
print(f"TVT range: [{submission['tvt'].min():.3f}, {submission['tvt'].max():.3f}]")
print('Diagnostics: run_diagnostics_physical_pf_9150_repeat.json')
print(submission.head())
